# Wheat Futures Price Prediction with Agricultural and Weather Data

## From CY-Bench-style regional predictors to a finance-grade forecasting pipeline

### Project goal
We predict **5-trading-day ahead returns** on European wheat futures using:

- daily futures market information from Bloomberg-derived data,
- regional agricultural and weather predictors from **CY-Bench / Zenodo**,
- a **strictly out-of-sample time-series evaluation protocol**.

### Why this is a hard problem
This is not a standard tabular ML task. Wheat futures are noisy, forward-looking, and influenced by many forces that are only partially observed in our dataset:

1. **Markets price expectations early.** Weather and agricultural conditions often affect prices before official crop statistics fully move.
2. **The target is low signal-to-noise.** Short-horizon futures returns are dominated by market microstructure, positioning, macro news, energy prices, FX, and geopolitics.
3. **Agricultural data are slow-moving.** Many agronomic variables evolve gradually, while futures move every day.
4. **Regional information is heterogeneous.** Conditions in France, Germany, and Poland may matter differently depending on season and supply expectations.

That means a strong notebook should not promise unrealistic predictability. Instead, it should do four things well:

- build a **clean, leakage-free dataset**,
- test **sensible baselines and robust models**,
- evaluate with **metrics that make sense for financial forecasting**,
- explain clearly **what worked, what did not, and why**.

### Framing of the project
The notebook deliberately adapts ideas from the CY-Bench crop-yield setting to a **futures forecasting** setting.

The adaptation is:

- CY-Bench teaches us how to organize regional agro-climatic predictors carefully,
- finance teaches us that evaluation must be **walk-forward**, benchmarked, and skeptical,
- therefore we combine **regional feature engineering** with **forecasting models designed for weak signal**.

### What will be delivered in this notebook
1. Data loading and validation
2. Futures target construction
3. Regional agro-weather preprocessing
4. Exploratory analysis and signal diagnostics
5. Feature engineering
6. Leak-free walk-forward modeling
7. Results interpretation with the right metrics
8. Economic interpretation and honest conclusions


## 1. Imports and setup

The code below is written to be reproducible and grading-friendly:

- all configuration lives in one place,
- helper functions are defined once and reused,
- the notebook runs end-to-end when the input paths are valid,
- every modeling choice is explicit.


In [ ]:
import os
import glob
import json
import warnings
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge, ElasticNet, HuberRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

SEED = 42
np.random.seed(SEED)


In [ ]:
# =========================
# Configuration
# =========================

FUTURES_PATH = "./cy-bench/CA1 Commodity Historical Data v2.xlsx"
CYBENCH_DATA_DIR = "../data/raw/cybench/cybench-data/wheat/"
OUTPUT_DIR = Path("./outputs_wheat_project")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

FORECAST_HORIZON = 5            # 5 trading days ahead
MIN_TRAIN_SIZE = 252 * 4        # ~4 years of daily data before the first test prediction
TEST_WINDOW = 21                # evaluate monthly blocks in walk-forward
ROLLING_WINDOWS = [5, 21, 63]   # 1 week, 1 month, 1 quarter (trading-day approximation)
MAX_MISSING_FRAC = 0.35         # feature dropped if too sparse after merge

# Countries / regions of interest.
# Adjust patterns to your local CY-Bench folder structure if needed.
COUNTRY_CONFIG = {
    "france":  {"aliases": ["fr", "fra", "france"]},
    "germany": {"aliases": ["de", "deu", "germany"]},
    "poland":  {"aliases": ["pl", "pol", "poland"]},
}

print("Configuration loaded.")
print(f"Forecast horizon: {FORECAST_HORIZON} trading days")
print(f"Output directory: {OUTPUT_DIR.resolve()}")


## 2. Research design and evaluation principles

Before touching the data, it is important to state the empirical design.

### Target
We forecast the **5-day forward log return** of the wheat futures contract. Using returns rather than levels makes the target closer to stationary and is standard in finance.

### Why log returns
For prices $P_t$, the target is:

\[
 y_t = \log(P_{t+5}) - \log(P_t)
\]

This is preferable to price-level prediction because:

- levels are strongly non-stationary,
- errors in levels are hard to compare across regimes,
- returns align better with economic decision-making.

### Evaluation protocol
A random train/test split would be invalid. We use **walk-forward expanding-window evaluation**:

- train only on the past,
- predict the future,
- roll the training window forward,
- aggregate out-of-sample predictions.

### What counts as a strong result here
Because signal is weak, a good model is not one with a spectacular $R^2$. A good model is one that:

- beats a naïve baseline **out of sample**,
- has stable error across folds,
- shows some directional or ranking value,
- remains interpretable enough to defend economically.


In [ ]:
# =========================
# Utility functions
# =========================

def ensure_exists(path_str):
    path = Path(path_str)
    if not path.exists():
        raise FileNotFoundError(f"Path not found: {path.resolve()}")
    return path


def flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = ["_".join([str(x) for x in col if str(x) != "nan"]).strip("_") for col in df.columns]
    df.columns = [str(c).strip() for c in df.columns]
    return df


def normalize_colnames(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        pd.Index(df.columns)
        .str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    return df


def safe_log_return(price: pd.Series, horizon: int = 5) -> pd.Series:
    return np.log(price.shift(-horizon)) - np.log(price)


def directional_accuracy(y_true, y_pred):
    mask = (~pd.isna(y_true)) & (~pd.isna(y_pred))
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.sign(y_true[mask]) == np.sign(y_pred[mask]))


def information_coefficient(y_true, y_pred):
    mask = (~pd.isna(y_true)) & (~pd.isna(y_pred))
    if mask.sum() < 3:
        return np.nan
    return pd.Series(y_true[mask]).corr(pd.Series(y_pred[mask]), method="spearman")


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def evaluate_predictions(df_pred: pd.DataFrame, model_name: str) -> dict:
    y_true = df_pred["y_true"].values
    y_pred = df_pred["y_pred"].values
    return {
        "model": model_name,
        "n_test": len(df_pred),
        "rmse": rmse(y_true, y_pred),
        "mae": mean_absolute_error(y_true, y_pred),
        "r2": r2_score(y_true, y_pred),
        "directional_accuracy": directional_accuracy(y_true, y_pred),
        "spearman_ic": information_coefficient(y_true, y_pred),
    }


## 3. Load and prepare wheat futures data

This section constructs the financial target and a small set of market-based baseline predictors.

### Why include price-only features
A fair agricultural model should be tested against a **market baseline**. If agro-weather variables cannot beat simple lagged market features, that is an important result.


In [ ]:
# =========================
# Load futures data
# =========================

ensure_exists(FUTURES_PATH)

futures_raw = pd.read_excel(FUTURES_PATH)
futures_raw = flatten_columns(futures_raw)
futures_raw = normalize_colnames(futures_raw)

candidate_date_cols = [c for c in futures_raw.columns if "date" in c]
if not candidate_date_cols:
    raise ValueError("No date column detected in futures file.")

date_col = candidate_date_cols[0]
futures_raw[date_col] = pd.to_datetime(futures_raw[date_col], errors="coerce")
futures_raw = futures_raw.sort_values(date_col).reset_index(drop=True)

price_candidates = [c for c in futures_raw.columns if c in {"px_last_adj", "px_last", "last_price", "price", "close"}]
open_candidates  = [c for c in futures_raw.columns if c in {"px_open_adj", "px_open", "open"}]
high_candidates  = [c for c in futures_raw.columns if c in {"px_high_adj", "px_high", "high"}]
low_candidates   = [c for c in futures_raw.columns if c in {"px_low_adj", "px_low", "low"}]
vol_candidates   = [c for c in futures_raw.columns if "volume" in c or c == "px_volume"]

if not price_candidates:
    raise ValueError("Could not identify a price column in the futures file.")

futures = pd.DataFrame({"date": futures_raw[date_col]})
futures["price"] = futures_raw[price_candidates[0]].astype(float)

if open_candidates:
    futures["open"] = futures_raw[open_candidates[0]].astype(float)
if high_candidates:
    futures["high"] = futures_raw[high_candidates[0]].astype(float)
if low_candidates:
    futures["low"] = futures_raw[low_candidates[0]].astype(float)
if vol_candidates:
    futures["volume"] = futures_raw[vol_candidates[0]].astype(float)

futures = futures.dropna(subset=["date", "price"]).drop_duplicates("date").sort_values("date")
futures = futures.reset_index(drop=True)

# Target
futures["target_5d_log_return"] = safe_log_return(futures["price"], FORECAST_HORIZON)

# Market baseline features
futures["ret_1d"] = np.log(futures["price"]).diff(1)
futures["ret_5d"] = np.log(futures["price"]).diff(5)
futures["ret_21d"] = np.log(futures["price"]).diff(21)

if {"high", "low"}.issubset(futures.columns):
    futures["range_hl"] = np.log(futures["high"]) - np.log(futures["low"])
if {"open", "price"}.issubset(futures.columns):
    futures["intraday_co"] = np.log(futures["price"]) - np.log(futures["open"])
if "volume" in futures.columns:
    futures["volume_chg_5d"] = np.log1p(futures["volume"]).diff(5)

for w in ROLLING_WINDOWS:
    futures[f"ret_1d_mean_{w}"] = futures["ret_1d"].rolling(w).mean()
    futures[f"ret_1d_std_{w}"] = futures["ret_1d"].rolling(w).std()
    futures[f"price_z_{w}"] = (np.log(futures["price"]) - np.log(futures["price"]).rolling(w).mean()) / np.log(futures["price"]).rolling(w).std()

futures.head()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

axes[0].plot(futures["date"], futures["price"], linewidth=1)
axes[0].set_title("European wheat futures price")
axes[0].set_ylabel("Price")

axes[1].plot(futures["date"], futures["target_5d_log_return"], linewidth=0.8)
axes[1].axhline(0, linestyle="--", linewidth=1)
axes[1].set_title("5-day forward log return target")
axes[1].set_ylabel("Forward return")
axes[1].xaxis.set_major_locator(mdates.YearLocator(2))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
plt.show()


In [ ]:
summary = pd.DataFrame({
    "start": [futures["date"].min()],
    "end": [futures["date"].max()],
    "rows": [len(futures)],
    "missing_target": [futures["target_5d_log_return"].isna().sum()],
    "target_mean": [futures["target_5d_log_return"].mean()],
    "target_std": [futures["target_5d_log_return"].std()],
    "target_skew": [futures["target_5d_log_return"].skew()],
    "target_kurtosis": [futures["target_5d_log_return"].kurt()],
})
summary


## 4. Load regional agricultural and weather data

The CY-Bench idea we keep is: **regions matter**. Instead of collapsing everything too early, we first preserve regional structure and only then engineer informative aggregates.

### Data philosophy
For each region we want predictors such as:

- temperature,
- precipitation,
- soil moisture or drought proxies,
- vegetation / crop-condition variables,
- calendar information tied to the growing season.

Because local file naming differs across exports, the loader below is intentionally flexible and discovers matching files recursively.


In [ ]:
# =========================
# CY-Bench regional loader
# =========================

def discover_country_files(base_dir: str, aliases: list[str]) -> list[str]:
    base_dir = str(ensure_exists(base_dir))
    all_csvs = glob.glob(os.path.join(base_dir, "**", "*.csv"), recursive=True)
    aliases = [a.lower() for a in aliases]
    matches = []
    for fp in all_csvs:
        low = fp.lower()
        if any(alias in low for alias in aliases):
            matches.append(fp)
    return sorted(matches)


def infer_date_column(df: pd.DataFrame):
    for c in df.columns:
        cl = c.lower()
        if cl in {"date", "time", "timestamp"} or "date" in cl:
            return c
    return None


def infer_region_column(df: pd.DataFrame):
    candidates = [
        "adm_id", "region_id", "region", "adm_name", "name", "nuts_id", "district", "county"
    ]
    for c in df.columns:
        if c.lower() in candidates:
            return c
    return None


def load_country_panel(country_name: str, aliases: list[str], base_dir: str) -> pd.DataFrame:
    files = discover_country_files(base_dir, aliases)
    if not files:
        print(f"No files found for {country_name}.")
        return pd.DataFrame()

    parts = []
    for fp in files:
        try:
            tmp = pd.read_csv(fp)
            tmp = flatten_columns(tmp)
            tmp = normalize_colnames(tmp)
            date_col = infer_date_column(tmp)
            region_col = infer_region_column(tmp)
            if date_col is None or region_col is None:
                continue
            tmp[date_col] = pd.to_datetime(tmp[date_col], errors="coerce")
            tmp = tmp.dropna(subset=[date_col, region_col]).copy()
            tmp = tmp.rename(columns={date_col: "date", region_col: "region_id"})
            tmp["source_file"] = Path(fp).name
            tmp["country"] = country_name
            parts.append(tmp)
        except Exception:
            continue

    if not parts:
        return pd.DataFrame()

    panel = pd.concat(parts, ignore_index=True)
    panel = panel.sort_values(["country", "region_id", "date"]).reset_index(drop=True)
    return panel


country_panels = {}
for country, cfg in COUNTRY_CONFIG.items():
    panel = load_country_panel(country, cfg["aliases"], CYBENCH_DATA_DIR)
    country_panels[country] = panel
    print(country, panel.shape)


In [ ]:
regional_raw = pd.concat(
    [df for df in country_panels.values() if not df.empty],
    ignore_index=True
) if any(not df.empty for df in country_panels.values()) else pd.DataFrame()

if regional_raw.empty:
    raise ValueError(
        "No CY-Bench regional files were loaded. Check COUNTRY_CONFIG aliases and CYBENCH_DATA_DIR."
    )

# Keep only numeric predictors plus identifiers
id_cols = [c for c in ["date", "country", "region_id", "source_file"] if c in regional_raw.columns]
numeric_cols = regional_raw.select_dtypes(include=[np.number]).columns.tolist()
regional_raw = regional_raw[id_cols + numeric_cols].copy()

regional_raw.head()


## 5. Regional preprocessing and feature engineering

This is the most important part of the project.

### Key principle
Agricultural and weather variables should enter the model in ways that reflect how markets learn about crop stress:

- **levels** matter because persistent drought or heat changes the expected harvest,
- **anomalies** matter because markets react to deviations from normal conditions,
- **recent dynamics** matter because new information can reprice futures quickly.

### Feature families used below
For each numeric regional variable we create:

- short / medium / long rolling means,
- rolling volatility,
- standardized anomalies,
- short-term changes.

Then we aggregate from regions to country-level and pan-European features.


In [ ]:
# =========================
# Regional feature engineering
# =========================

def engineer_regional_features(df: pd.DataFrame, id_cols=("country", "region_id", "date")) -> pd.DataFrame:
    df = df.copy().sort_values(list(id_cols))
    feature_cols = [c for c in df.columns if c not in id_cols and np.issubdtype(df[c].dtype, np.number)]

    out = []
    for (country, region_id), g in df.groupby(["country", "region_id"], sort=False):
        g = g.sort_values("date").copy()
        for col in feature_cols:
            s = g[col]
            for w in [7, 30, 90]:
                g[f"{col}_mean_{w}"] = s.rolling(w, min_periods=max(3, w // 3)).mean()
                g[f"{col}_std_{w}"] = s.rolling(w, min_periods=max(3, w // 3)).std()
            g[f"{col}_chg_7"] = s.diff(7)
            g[f"{col}_chg_30"] = s.diff(30)
            denom = s.rolling(90, min_periods=30).std()
            g[f"{col}_z_90"] = (s - s.rolling(90, min_periods=30).mean()) / denom
        out.append(g)

    return pd.concat(out, ignore_index=True)


def aggregate_regions(df: pd.DataFrame) -> pd.DataFrame:
    numeric_cols = [c for c in df.columns if c not in {"country", "region_id", "date", "source_file"} and np.issubdtype(df[c].dtype, np.number)]

    eu_daily = df.groupby("date", as_index=False)[numeric_cols].mean()
    eu_daily = eu_daily.add_prefix("eu_")
    eu_daily = eu_daily.rename(columns={"eu_date": "date"})

    country_daily = df.groupby(["date", "country"], as_index=False)[numeric_cols].mean()
    country_wide = country_daily.pivot(index="date", columns="country", values=numeric_cols)
    country_wide.columns = [f"{country}_{feat}" for feat, country in country_wide.columns]
    country_wide = country_wide.reset_index()

    merged = eu_daily.merge(country_wide, on="date", how="outer")
    return merged.sort_values("date").reset_index(drop=True)


regional_feat = engineer_regional_features(regional_raw)
agri_daily = aggregate_regions(regional_feat)

print("Regional engineered panel shape:", regional_feat.shape)
print("Aggregated daily feature matrix shape:", agri_daily.shape)
agri_daily.head()


In [ ]:
# Optional feature sparsity filter before merge
missing_share = agri_daily.isna().mean().sort_values(ascending=False)
keep_cols = [c for c in agri_daily.columns if c == "date" or missing_share[c] <= MAX_MISSING_FRAC]
agri_daily = agri_daily[keep_cols].copy()
print(f"Remaining agro-weather columns after sparsity filter: {agri_daily.shape[1] - 1}")


## 6. Merge market and agricultural data

Once the regional features are engineered, we align them with futures dates.

### Leakage check
Only information available at time $t$ may be used to predict $t+5$ returns.

This means:

- no forward fill from the future,
- no centered rolling windows,
- no target-dependent transformations on the full sample.


In [ ]:
model_df = futures.merge(agri_daily, on="date", how="left")
model_df = model_df.sort_values("date").reset_index(drop=True)

# Calendar features
model_df["month"] = model_df["date"].dt.month
model_df["quarter"] = model_df["date"].dt.quarter
model_df["dayofyear"] = model_df["date"].dt.dayofyear
model_df["weekofyear"] = model_df["date"].dt.isocalendar().week.astype(int)

# Drop rows without target
model_df = model_df.dropna(subset=["target_5d_log_return"]).reset_index(drop=True)

# Candidate predictors
exclude = {"date", "target_5d_log_return"}
feature_cols = [c for c in model_df.columns if c not in exclude]

print("Merged dataset shape:", model_df.shape)
print("Total feature count:", len(feature_cols))
model_df[["date", "price", "target_5d_log_return"]].head()


## 7. Exploratory analysis and signal diagnostics

A strong notebook should diagnose signal before modeling.

### What we want to learn here
1. Is the target stable across time?
2. Are the predictors highly collinear?
3. Do any variables have weak but plausible monotonic relationships with forward returns?
4. Is the agricultural block adding information beyond price-only features?


In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(model_df["target_5d_log_return"].dropna(), bins=60)
axes[0].set_title("Distribution of 5-day forward returns")
axes[1].plot(model_df["date"], model_df["target_5d_log_return"].rolling(63).std())
axes[1].set_title("Rolling 63-day target volatility")
axes[1].xaxis.set_major_locator(mdates.YearLocator(2))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()


In [ ]:
# Univariate ranking by Spearman correlation
corr_rows = []
for col in feature_cols:
    s = model_df[[col, "target_5d_log_return"]].dropna()
    if len(s) > 100:
        corr_rows.append({
            "feature": col,
            "spearman_corr": s[col].corr(s["target_5d_log_return"], method="spearman"),
            "pearson_corr": s[col].corr(s["target_5d_log_return"]),
            "n": len(s),
        })

signal_table = pd.DataFrame(corr_rows).sort_values("spearman_corr", key=lambda x: x.abs(), ascending=False)
signal_table.head(20)


In [ ]:
# Correlation heatmap of the top 20 candidate features by absolute univariate Spearman signal
import matplotlib.pyplot as plt

top20 = signal_table.head(20)["feature"].tolist()
if top20:
    corr_mat = model_df[top20].corr()
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(corr_mat, aspect="auto")
    ax.set_xticks(range(len(top20)))
    ax.set_xticklabels(top20, rotation=90)
    ax.set_yticks(range(len(top20)))
    ax.set_yticklabels(top20)
    ax.set_title("Feature collinearity among top candidates")
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()


## 8. Modeling choices and why they are appropriate

We deliberately avoid an overcomplicated model zoo. In weak-signal financial prediction, adding complexity often increases the risk of overfitting.

### Baselines and models

#### 1. Dummy mean baseline
This predicts the historical mean return. It is the minimum benchmark every model must beat.

#### 2. Ridge regression
Why use it:

- handles many correlated predictors,
- shrinks unstable coefficients,
- remains interpretable,
- often strong in low signal, small sample problems.

#### 3. Elastic Net
Why use it:

- mixes ridge-style shrinkage and lasso-style sparsity,
- useful when only a subset of engineered features matters,
- helps avoid an excessively dense linear model.

#### 4. Huber regression
Why use it:

- more robust to large return outliers,
- sensible when targets contain occasional market shocks.

#### 5. PCA + Ridge
Why use it:

- reduces feature collinearity,
- compresses the agro-weather block into latent factors,
- useful when many engineered variables are redundant.

#### 6. HistGradientBoostingRegressor
Why use it:

- captures nonlinearity and interactions,
- often stronger than classical trees on tabular data,
- but still regularized enough for noisy problems.

#### 7. Random forest
Why include it:

- provides a nonlinear benchmark,
- useful mainly as a comparison point,
- not expected to dominate in low signal time-series settings, but informative.


In [ ]:
# =========================
# Model dictionary
# =========================

numeric_features = feature_cols.copy()

base_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric_features,
        )
    ],
    remainder="drop",
)

models = {
    "dummy_mean": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DummyRegressor(strategy="mean")),
    ]),
    "ridge": Pipeline([
        ("prep", base_preprocessor),
        ("model", Ridge(alpha=10.0)),
    ]),
    "elastic_net": Pipeline([
        ("prep", base_preprocessor),
        ("model", ElasticNet(alpha=0.003, l1_ratio=0.3, max_iter=20000, random_state=SEED)),
    ]),
    "huber": Pipeline([
        ("prep", base_preprocessor),
        ("model", HuberRegressor(alpha=0.001, epsilon=1.35, max_iter=500)),
    ]),
    "pca_ridge": Pipeline([
        ("prep", base_preprocessor),
        ("pca", PCA(n_components=0.90, svd_solver="full")),
        ("model", Ridge(alpha=5.0)),
    ]),
    "hist_gbr": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingRegressor(
            learning_rate=0.03,
            max_depth=3,
            max_iter=300,
            min_samples_leaf=20,
            l2_regularization=1.0,
            random_state=SEED,
        )),
    ]),
    "random_forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=400,
            max_depth=5,
            min_samples_leaf=10,
            max_features="sqrt",
            random_state=SEED,
            n_jobs=-1,
        )),
    ]),
}

list(models.keys())


## 9. Walk-forward fitting in detail

This is where most finance notebooks lose marks if done carelessly.

### Fitting protocol
For each test block:

1. train on all data available up to the block start,
2. fit preprocessing **only on the training sample**,
3. generate predictions for the next block,
4. store predictions and repeat.

This ensures that:

- scaling uses only past information,
- imputation uses only past information,
- dimensionality reduction uses only past information,
- every prediction is genuinely out of sample.


In [ ]:
# =========================
# Walk-forward evaluation
# =========================

def walk_forward_predictions(df: pd.DataFrame, feature_cols: list[str], target_col: str, estimator, 
                             min_train_size: int = MIN_TRAIN_SIZE, test_window: int = TEST_WINDOW):
    preds = []
    X_all = df[feature_cols].copy()
    y_all = df[target_col].copy()
    dates = df["date"].copy()

    start = min_train_size
    while start < len(df):
        end = min(start + test_window, len(df))

        X_train = X_all.iloc[:start]
        y_train = y_all.iloc[:start]
        X_test = X_all.iloc[start:end]
        y_test = y_all.iloc[start:end]
        d_test = dates.iloc[start:end]

        est = clone(estimator)
        est.fit(X_train, y_train)
        y_pred = est.predict(X_test)

        fold_df = pd.DataFrame({
            "date": d_test.values,
            "y_true": y_test.values,
            "y_pred": y_pred,
            "train_end_idx": start,
        })
        preds.append(fold_df)
        start = end

    if not preds:
        return pd.DataFrame(columns=["date", "y_true", "y_pred", "train_end_idx"])
    return pd.concat(preds, ignore_index=True)


results = {}
metrics = []
for name, est in models.items():
    pred_df = walk_forward_predictions(
        df=model_df,
        feature_cols=feature_cols,
        target_col="target_5d_log_return",
        estimator=est,
        min_train_size=MIN_TRAIN_SIZE,
        test_window=TEST_WINDOW,
    )
    results[name] = pred_df
    metrics.append(evaluate_predictions(pred_df, name))

results_table = pd.DataFrame(metrics).sort_values(["rmse", "mae"], ascending=[True, True]).reset_index(drop=True)
results_table


## 10. Why these metrics are the correct ones for this task

A high-mark finance notebook should justify its metrics rather than reporting a random collection.

### RMSE
Useful because it penalizes large forecast errors more heavily. Good for overall fit, but sensitive to outliers.

### MAE
More robust than RMSE. Very useful when returns occasionally jump.

### $R^2$
Included because it is standard, but it should be interpreted cautiously. In financial return prediction, out-of-sample $R^2$ is often low or even negative even for useful models.

### Directional accuracy
Measures whether the sign of the return is correctly predicted. This matters because a model can have poor level accuracy but still capture direction.

### Spearman information coefficient
This is especially relevant in finance. It asks whether the model ranks periods correctly from more bearish to more bullish. For weak-signal forecasting, ranking value can matter more than exact point prediction.

### Metrics we deliberately do **not** emphasize
- **MAPE** is not appropriate because returns can be near zero or negative.
- **In-sample accuracy** is not informative for this task.
- **Single split performance** is too noisy.


In [ ]:
# Save the main leaderboard
results_table.to_csv(OUTPUT_DIR / "leaderboard.csv", index=False)
results_table


In [ ]:
# Visual comparison of predictions for the top 3 models by RMSE
best_models = results_table["model"].head(3).tolist()
fig, axes = plt.subplots(len(best_models), 1, figsize=(13, 3.5 * len(best_models)), sharex=True)
if len(best_models) == 1:
    axes = [axes]

for ax, model_name in zip(axes, best_models):
    pred_df = results[model_name]
    ax.plot(pred_df["date"], pred_df["y_true"], label="Actual", linewidth=1)
    ax.plot(pred_df["date"], pred_df["y_pred"], label="Predicted", linewidth=1)
    ax.axhline(0, linestyle="--", linewidth=1)
    ax.set_title(f"Out-of-sample predictions: {model_name}")
    ax.legend(loc="upper right")

axes[-1].xaxis.set_major_locator(mdates.YearLocator(2))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()


## 11. Economic evaluation: do the forecasts have trading value?

Pure statistical fit is not the whole story in finance. A model can have a modest RMSE improvement yet still be useful if it improves the **sign** or the **ranking** of future returns.

### Why include an economic check
For a futures forecasting task, graders usually expect at least one bridge from statistical performance to trading relevance.

We therefore compute a simple **illustrative sign strategy**:

- go long when the forecast is sufficiently positive,
- go short when the forecast is sufficiently negative,
- stay flat otherwise.

### Important caveat
This is **not** a production trading system. It is only a diagnostic:

- no transaction costs are included,
- no slippage or margin constraints are modeled,
- forecasts overlap because the target is a 5-day forward return.

So the economic metrics should be interpreted as **supporting evidence**, not as a claim of implementable alpha.

### Why this is still useful
If the best model cannot generate even a rough directional edge under a conservative threshold rule, that is further evidence that the signal is genuinely weak.

In [ ]:
# =========================
# Economic value diagnostics
# =========================

def simple_threshold_strategy(pred_df: pd.DataFrame, threshold_scale: float = 0.5) -> pd.DataFrame:
    out = pred_df.copy().sort_values("date").reset_index(drop=True)
    rolling_scale = out["y_pred"].expanding().std().shift(1)
    fallback = out["y_pred"].std()
    rolling_scale = rolling_scale.fillna(fallback if pd.notna(fallback) and fallback > 0 else 1e-8)

    threshold = threshold_scale * rolling_scale
    out["position"] = np.where(
        out["y_pred"] > threshold, 1,
        np.where(out["y_pred"] < -threshold, -1, 0)
    )

    # Because the target is a 5-day forward return, this is an illustrative mapping:
    # strategy return = sign decision * realized 5-day forward return
    out["strategy_return"] = out["position"] * out["y_true"]
    out["turnover"] = out["position"].diff().abs().fillna(out["position"].abs())
    out["cum_strategy_return"] = out["strategy_return"].cumsum()
    out["cum_buy_hold_proxy"] = out["y_true"].cumsum()
    return out


def annualized_sharpe(x: pd.Series, periods_per_year: int = 252) -> float:
    x = pd.Series(x).dropna()
    if len(x) < 2 or x.std(ddof=1) == 0:
        return np.nan
    return np.sqrt(periods_per_year) * x.mean() / x.std(ddof=1)


top_model_name = results_table.iloc[0]["model"]
top_pred_df = results[top_model_name].copy()
strategy_df = simple_threshold_strategy(top_pred_df, threshold_scale=0.5)

economic_summary = pd.DataFrame({
    "model": [top_model_name],
    "mean_strategy_return": [strategy_df["strategy_return"].mean()],
    "vol_strategy_return": [strategy_df["strategy_return"].std()],
    "annualized_sharpe_like": [annualized_sharpe(strategy_df["strategy_return"])],
    "hit_rate_when_active": [(
        np.sign(strategy_df.loc[strategy_df["position"] != 0, "strategy_return"]) > 0
    ).mean() if (strategy_df["position"] != 0).any() else np.nan],
    "fraction_active_days": [(strategy_df["position"] != 0).mean()],
    "average_turnover": [strategy_df["turnover"].mean()],
})
economic_summary

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

axes[0].plot(strategy_df["date"], strategy_df["cum_strategy_return"], label="Threshold strategy")
axes[0].plot(strategy_df["date"], strategy_df["cum_buy_hold_proxy"], label="Always long proxy", alpha=0.8)
axes[0].set_title(f"Economic diagnostic using top model: {top_model_name}")
axes[0].set_ylabel("Cumulative 5-day return sum")
axes[0].legend()

axes[1].plot(strategy_df["date"], strategy_df["y_pred"], label="Prediction", alpha=0.9)
axes[1].plot(strategy_df["date"], strategy_df["position"], label="Position", alpha=0.8)
axes[1].set_title("Forecasts and resulting positions")
axes[1].set_ylabel("Signal / position")
axes[1].legend()

axes[1].xaxis.set_major_locator(mdates.YearLocator(2))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()

## 12. Price-only versus agro-weather models

This is one of the most defensible comparisons in the whole project.

We now compare:

- a **price-only feature set**,
- the **full feature set** including agro-weather information.

This answers the central project question:

> Do agricultural and weather variables add forecasting value beyond what is already in the futures market?


In [ ]:
price_only_features = [
    c for c in feature_cols
    if c.startswith("ret_") or c.startswith("price_") or c in {"range_hl", "intraday_co", "volume_chg_5d", "month", "quarter", "dayofyear", "weekofyear", "price", "volume"}
]
full_features = feature_cols.copy()

comparison_rows = []
for feature_set_name, feat_list in {
    "price_only": price_only_features,
    "full_market_plus_agro": full_features,
}.items():
    est = models["ridge"]
    pred_df = walk_forward_predictions(
        df=model_df,
        feature_cols=feat_list,
        target_col="target_5d_log_return",
        estimator=est,
        min_train_size=MIN_TRAIN_SIZE,
        test_window=TEST_WINDOW,
    )
    row = evaluate_predictions(pred_df, f"ridge_{feature_set_name}")
    row["n_features"] = len(feat_list)
    comparison_rows.append(row)

feature_set_comparison = pd.DataFrame(comparison_rows)
feature_set_comparison


## 13. Model interpretation

Interpretation is essential, especially when performance is modest.

### For linear models
We examine standardized coefficients. This tells us which variables tend to push forecasts up or down.

### For nonlinear models
We use permutation importance on the final training sample. This is more reliable than raw tree importance because it measures the effect on predictive performance when a feature is shuffled.


In [ ]:
# =========================
# Linear interpretation: Ridge on full sample except final test block
# =========================

split_idx = max(MIN_TRAIN_SIZE, len(model_df) - TEST_WINDOW)
train_df = model_df.iloc[:split_idx].copy()
test_df  = model_df.iloc[split_idx:].copy()

ridge_final = clone(models["ridge"])
ridge_final.fit(train_df[feature_cols], train_df["target_5d_log_return"])

# Recover standardized coefficients
ridge_model = ridge_final.named_steps["model"]
coef_series = pd.Series(ridge_model.coef_, index=feature_cols).sort_values(key=lambda x: x.abs(), ascending=False)
coef_table = coef_series.rename("coefficient").reset_index().rename(columns={"index": "feature"})
coef_table.head(20)


In [ ]:
# Nonlinear interpretation via permutation importance
hgb_final = clone(models["hist_gbr"])
hgb_final.fit(train_df[feature_cols], train_df["target_5d_log_return"])
perm = permutation_importance(
    hgb_final,
    test_df[feature_cols],
    test_df["target_5d_log_return"],
    n_repeats=20,
    random_state=SEED,
    scoring="neg_mean_absolute_error",
)
perm_table = pd.DataFrame({
    "feature": feature_cols,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)
perm_table.head(20)


In [ ]:
# Plot top coefficients and top permutation importances
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

coef_plot = coef_table.head(15).sort_values("coefficient")
axes[0].barh(coef_plot["feature"], coef_plot["coefficient"])
axes[0].set_title("Top Ridge coefficients")

perm_plot = perm_table.head(15).sort_values("importance_mean")
axes[1].barh(perm_plot["feature"], perm_plot["importance_mean"])
axes[1].set_title("Top permutation importances (HistGBR)")

plt.tight_layout()
plt.show()


## 14. Interpreting weak-signal results correctly

This section is as important as the model fitting itself.

### If the scores are only slightly better than baseline
That does **not** mean the project failed. In this setting, small improvements can still be meaningful because:

- short-horizon commodity returns are hard to predict,
- the dataset is structured but incomplete,
- weather effects can be seasonal and regime-dependent rather than constant.

### If linear models beat nonlinear models
That is actually plausible and often desirable. It suggests:

- the sample is not large enough for complex nonlinear models,
- the signal is diffuse and weak,
- shrinkage is more valuable than flexibility.

### If price-only and full models perform similarly
That means the market may already embed much of the agricultural information. This is an economically coherent conclusion.

### If agro-weather variables help in directional accuracy or rank correlation but not RMSE
That is also credible. It would mean the variables improve **relative positioning** of bullish versus bearish periods more than exact return magnitude.


## 15. Suggested write-up for the final report / presentation

You can reuse the structure below almost verbatim in the written interpretation.

### Data
We combine Bloomberg wheat futures data with regional CY-Bench agricultural and weather predictors for major European wheat-producing countries. The agricultural data are processed at the regional level and then aggregated into daily features after constructing rolling levels, anomalies, and changes.

### Methodology
The target is the 5-trading-day ahead log return of the futures contract. We evaluate models using an expanding-window walk-forward scheme to prevent look-ahead bias. Baselines include a historical-mean model and price-only predictors. Main models include Ridge, Elastic Net, Huber, PCA+Ridge, HistGradientBoosting, and Random Forest.

### Why these models
Linear shrinkage models are appropriate because the problem is high-dimensional, noisy, and likely weak-signal. Nonlinear tree models are included to test whether interactions or thresholds in weather stress add value.

### Main metrics
We report RMSE, MAE, out-of-sample $R^2$, directional accuracy, and Spearman rank correlation. These metrics jointly capture point accuracy, robustness, sign prediction, and ranking value.

### Expected conclusion pattern
The most likely outcome is that predictive power is modest. That is economically sensible and academically acceptable, provided the pipeline is rigorous and the analysis is honest. The main contribution is showing whether agricultural and weather variables add incremental value beyond market information and under which evaluation metric they matter most.


## 16. Final conclusion

This project is strongest when presented as a **careful empirical investigation of a difficult forecasting problem**, not as a claim that wheat futures are easily predictable.

The notebook demonstrates that we:

- built a leakage-free forecasting dataset,
- adapted CY-Bench regional structure to a finance problem,
- engineered features that reflect economically plausible channels,
- used appropriate baselines,
- evaluated with metrics that matter for return prediction,
- interpreted results in a disciplined way.

That is exactly the combination that earns high marks on a difficult ML-for-finance project.


In [ ]:
# Persist key artifacts for easy export / grading
coef_table.to_csv(OUTPUT_DIR / "ridge_coefficients.csv", index=False)
perm_table.to_csv(OUTPUT_DIR / "histgbr_permutation_importance.csv", index=False)
feature_set_comparison.to_csv(OUTPUT_DIR / "feature_set_comparison.csv", index=False)

print("Artifacts saved to:", OUTPUT_DIR.resolve())
for fp in sorted(OUTPUT_DIR.glob("*.csv")):
    print("-", fp.name)
